# 🎓 Educational Tutorial: VGG16 Architecture from Scratch

This notebook serves as a self-contained learning artifact detailing the history, design choices, mathematical intuition, and implementation of **VGG16** (2014) from first principles.

---

## 1. Historical Background

* **Title**: *Very Deep Convolutional Networks for Large-Scale Image Recognition*
* **Authors**: Karen Simonyan, Andrew Zisserman (Visual Geometry Group, University of Oxford)
* **Year**: 2014 (Presented at ICLR 2015)

### Historical Context
Following the success of AlexNet in 2012, researchers focused on improving model performance. Most approaches involved adjusting hyperparameters or adding ad-hoc layers. The Visual Geometry Group (VGG) at Oxford chose a systematic approach: they wanted to explore the impact of **network depth** (how deep a network could get) while using a extremely simple, uniform design language throughout the entire model.

## 2. Why the Previous CNN (AlexNet) was Insufficient

Although AlexNet proved the power of deep neural networks, its architecture was somewhat arbitrary:
1. **Large Conv Filters**: AlexNet used very large convolutional kernels in the early layers ($11 \times 11$ and $5 \times 5$). These filters are computationally expensive, possess a massive amount of parameters, and only perform a single non-linear projection.
2. **Irregular Design**: AlexNet has varying kernel sizes, strides, and layers that lack a cohesive architectural principle, making it difficult to scale or customize systematically.
3. **Parameter Inefficiency**: Large kernels have fewer non-linearities per parameter. Stacking smaller kernels can represent more complex non-linear decision boundaries with far fewer weights.

## 3. Mathematical Intuition: Stacking 3x3 Kernels

VGG's core idea is that a stack of **two** $3 \times 3$ convolutional layers has an effective receptive field of a single $5 \times 5$ layer, and a stack of **three** $3 \times 3$ conv layers has an effective receptive field of a single $7 \times 7$ layer.

Let's look at the parameter calculation. Let the input and output channel count be $C$:

* **Single $7 \times 7$ Convolution**:
  $$\text{Parameters} = 7 \times 7 \times C \times C = 49 C^2$$

* **Three stacked $3 \times 3$ Convolutions**:
  $$\text{Parameters} = 3 \times (3 \times 3 \times C \times C) = 27 C^2$$

### Advantages of Stacked Convolutions:
1. **Parameter Reduction**: The three stacked $3 \times 3$ filters use **45% fewer parameters** ($27 C^2$ vs $49 C^2$) compared to a single $7 \times 7$ layer.
2. **More Non-Linearities**: Stacking three layers introduces **three ReLU activations** instead of just one, allowing the network to learn far more complex discriminative features.

## 4. Layer-by-Layer Architectural Breakdown (VGG16)

VGG16 consists of 13 Convolutional layers and 3 Fully Connected layers, structured into 5 uniform blocks:

* **Block 1**: 2 × Conv(64, 3x3, pad=1) -> MaxPool(2x2, stride=2). Output size: $112 \times 112 \times 64$.
* **Block 2**: 2 × Conv(128, 3x3, pad=1) -> MaxPool(2x2, stride=2). Output size: $56 \times 56 \times 128$.
* **Block 3**: 3 × Conv(256, 3x3, pad=1) -> MaxPool(2x2, stride=2). Output size: $28 \times 28 \times 256$.
* **Block 4**: 3 × Conv(512, 3x3, pad=1) -> MaxPool(2x2, stride=2). Output size: $14 \times 14 \times 512$.
* **Block 5**: 3 × Conv(512, 3x3, pad=1) -> MaxPool(2x2, stride=2). Output size: $7 \times 7 \times 512$.
* **Classifier**: Flatten to $25,088$ nodes -> FC(4096) -> FC(4096) -> FC(10) outputs.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import pandas as pd
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
import time

print("PyTorch version:", torch.__version__)

In [ ]:
class VGG16FromScratch(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        
        # 1. Feature Extractor Blocks
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 5
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Adaptive Average Pooling
            nn.AdaptiveAvgPool2d((7, 7))
        )
        
        # 2. Classifier
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            
            nn.Linear(4096, num_classes)
        )
        
        self._initialize_weights()
        
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)
        return x
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

model = VGG16FromScratch()
print("Parameter Count:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
class NotebookEuroSATDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = self.root_dir / row["image_path"]
        image = Image.open(img_path).convert("RGB")
        label = row["label"]
        
        if self.transform:
            image = self.transform(image)
            
        return {
            "image": image,
            "label": label
        }

In [ ]:
PROJECT_ROOT = Path("../../")
csv_path = PROJECT_ROOT / "data/processed/train.csv"
val_csv_path = PROJECT_ROOT / "data/processed/validation.csv"
img_dir = PROJECT_ROOT / "data/raw/EuroSAT_RGB"

if not csv_path.exists():
    PROJECT_ROOT = Path(".").resolve().parents[1]
    csv_path = PROJECT_ROOT / "data/processed/train.csv"
    val_csv_path = PROJECT_ROOT / "data/processed/validation.csv"
    img_dir = PROJECT_ROOT / "data/raw/EuroSAT_RGB"

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

if csv_path.exists() and img_dir.exists():
    train_dataset = NotebookEuroSATDataset(csv_path, img_dir, train_transform)
    val_dataset = NotebookEuroSATDataset(val_csv_path, img_dir, val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)  # smaller batch size due to VGG size
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
    print(f"EuroSAT dataset splits loaded successfully.")
else:
    print("Warning: EuroSAT files not found! Mocking data loading.")
    class DummyDataset(Dataset):
        def __len__(self): return 32
        def __getitem__(self, idx): return {"image": torch.randn(3, 224, 224), "label": torch.randint(0, 10, (1,)).item()}
    train_loader = DataLoader(DummyDataset(), batch_size=4, shuffle=True)
    val_loader = DataLoader(DummyDataset(), batch_size=4, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

model = VGG16FromScratch().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

model.train()
start_time = time.time()
epoch_loss = 0.0
correct = 0
total = 0

print("Starting training verification step...")
max_batches = 3
for idx, batch in enumerate(train_loader):
    if idx >= max_batches: break
    images = batch["image"].to(device)
    labels = batch["label"].to(device)
    
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    epoch_loss += loss.item() * images.size(0)
    correct += (outputs.argmax(dim=1) == labels).sum().item()
    total += labels.size(0)
    
print(f"Quick epoch validation complete in {time.time() - start_time:.2f}s | Loss: {epoch_loss/total:.4f} | Acc: {correct/total*100:.2f}%")

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for idx, batch in enumerate(val_loader):
        if idx >= 2: break
        images = batch["image"].to(device)
        labels = batch["label"]
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu()
        
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, zero_division=0))

## 6. Discussion: Strengths & Weaknesses

### Strengths
1. **Architectural Simplicity**: The design relies solely on stacking $3 \times 3$ convolutions and $2 \times 2$ poolings in blocks, which is clean, easy to read, and modular.
2. **Strong Feature Representation**: Due to its depth, VGG16 learns hierarchical features (from simple edge representations to complex high-level geometry) extremely well. VGG features became the standard for image generation, neural style transfer, and object detection.

### Weaknesses
1. **Highly Computational**: VGG16 contains over **138 Million parameters** (compared to AlexNet's 60M). 
2. **Memory Heavy**: The early convolutional feature maps consume large amounts of activation memory, making it slow and difficult to train on smaller GPUs.
3. **Vanishing Gradient**: As network depth increases, gradients tend to become small during backpropagation, stalling the optimization of early layers.